
 Notebook 05 — Sistema de Respostas Contextuais

 Objetivo:
 Demonstrar um sistema que gera respostas contextuais de moda
 combinando:
   - Perfil da cliente (tamanhos, estilos, restrições)
   - Catálogo disponível (produtos normalizados)
   - Índice semântico (busca inteligente via embeddings)#


Introdução
Este notebook marca a segunda fase do nosso projeto de assistente virtual para a Curadobia. O nosso objetivo é evoluir o modelo de classificação de intenções, que atua como o cérebro do assistente. Através da análise e do processamento de dados de conversas com clientes, construímos e refinamos um modelo capaz de entender o que o cliente quer, transformando mensagens de texto em intenções claras, como "dúvida sobre preço" ou "pedido de sugestão de produto".

O principal foco desta fase é a maturidade do modelo. Isso significa não apenas buscar uma boa performance em métricas como F1 e acurácia, mas também garantir que todo o processo de construção e avaliação seja reproduzível, transparente e bem documentado. Nosso compromisso é que este caderno sirva como um "entregável" completo, com explicações claras para que qualquer pessoa consiga acompanhar a jornada de transformação dos dados até a publicação do modelo final.

In [23]:
# CÉLULA 2 — Setup e caminhos principais

import os, sys, json, pathlib, subprocess
from datetime import datetime
import pandas as pd
import numpy as np

# ---------------- Funções utilitárias ----------------
def hr(title=None, ch="="):
    """Imprime separador com título opcional."""
    print("\n" + ch*100)
    if title:
        print(title)
        print(ch*100)

def pp(x):
    """Pretty-print para JSON/dicts."""
    print(json.dumps(x, ensure_ascii=False, indent=2))

def resolve_repo_root(start: pathlib.Path) -> pathlib.Path:
    """Sobe diretórios até encontrar raiz provável do repositório."""
    for p in [start, *start.parents]:
        if ((p / "requirements.txt").exists() or (p / "README.md").exists() or (p / ".git").exists()) and (p / "code").exists():
            return p
    return start

# ---------------- Caminhos principais ----------------
CWD = pathlib.Path.cwd()
ROOT = resolve_repo_root(CWD)

DATA   = ROOT / "data"
CAT    = DATA / "catalog"
PROF   = DATA / "profiles"
INTENTS= DATA / "intents"
NB     = ROOT / "code" / "notebooks"
OUT    = NB / "outputs"
RUNS   = OUT / "runs"
MODELS = OUT / "models"
INDEX  = MODELS / "catalog_index"   # necessário em vários trechos

# Garante que pastas de saída existam
for p in [RUNS, MODELS, INDEX]:
    p.mkdir(parents=True, exist_ok=True)

# ---------------- Diagnóstico ----------------
hr("AMBIENTE / PATHS")
print("Python:", sys.version.split()[0])
print("CWD   :", CWD)
print("ROOT  :", ROOT)
print("DATA  :", DATA)
print("RUNS  :", RUNS)
print("MODELS:", MODELS)
print("INDEX :", INDEX)



AMBIENTE / PATHS
Python: 3.13.7
CWD   : c:\Users\win\Documents\GitHub\2025-2A-T07-CC11-G04\code\notebooks
ROOT  : c:\Users\win\Documents\GitHub\2025-2A-T07-CC11-G04
DATA  : c:\Users\win\Documents\GitHub\2025-2A-T07-CC11-G04\data
RUNS  : c:\Users\win\Documents\GitHub\2025-2A-T07-CC11-G04\code\notebooks\outputs\runs
MODELS: c:\Users\win\Documents\GitHub\2025-2A-T07-CC11-G04\code\notebooks\outputs\models
INDEX : c:\Users\win\Documents\GitHub\2025-2A-T07-CC11-G04\code\notebooks\outputs\models\catalog_index


In [24]:
# CÉLULA 3 — Carregar dataset unificado
# Este dataset reúne mensagens (Instagram, WhatsApp, etc.)
# e serve como base para intenções, clusters e análise semântica.

# Define o caminho para o arquivo CSV do dataset unificado.
ds_path = NB / "dataset" / "dataset_unificado.csv"
# Carrega o dataset em um DataFrame.
dfu = pd.read_csv(ds_path)

# Exibe informações básicas do DataFrame para verificação.
hr("Dataset unificado carregado")
print("Shape:", dfu.shape)
print("Colunas:", list(dfu.columns))
# Mostra as 10 primeiras linhas do DataFrame para inspeção.
dfu.head(10)


Dataset unificado carregado
Shape: (1633, 2)
Colunas: ['texto', 'intent']


,texto,intent
0,consiggo falar com atendente?,falar_com_humano
1,"veioo com defeito, como faço a devolução?",troca_devolucao
2,que calca maravilhosa,duvida_tamanho
3,a jaqueta branca uma pena eu nao conseguir com...,duvida_tamanho
4,ela a c reta e certa acredito que o tamanho m ...,outros
5,conjuntinho,outros
6,somos uma produtora de a udio visual e achamos...,duvida_tamanho
7,essa modelagem é mais enxuta? o camisa de visc...,tamanho_modelagem
8,boa tarde fabiana como vai passando para avisa...,saudacao
9,tem calça alfaiataria verde 40?,buscar_produto_por_nome


In [25]:
# CÉLULA 4 — Estatísticas básicas do dataset (robusta a nomes de colunas)

hr("Estatísticas básicas")

# 1) Descobrir coluna de TEXTO disponível
text_candidates = [
    "mensagem_clean", "texto_clean", "texto_ml",
    "mensagem", "texto", "message", "content", "utterance", "body"
]
cols_lower = {c.lower(): c for c in dfu.columns}
TEXT_COL = next((cols_lower[c] for c in text_candidates if c in cols_lower), None)

if TEXT_COL is None:
    raise KeyError(f"Não encontrei coluna de texto nas opções: {text_candidates}\n"
                   f"Colunas disponíveis: {list(dfu.columns)}")

# 2) Opcional: tentar achar uma versão *clean* já pronta; se não tiver, usa o texto bruto
clean_candidates = ["mensagem_clean", "texto_clean", "texto_ml"]
CLEAN_COL = next((cols_lower[c] for c in clean_candidates if c in cols_lower), TEXT_COL)

# 3) Contagens
print("Mensagens totais:", len(dfu))

# conta “texto não vazio” de forma segura (não nulo e não string vazia)
n_clean = (
    dfu[CLEAN_COL]
    .astype(str)
    .map(lambda s: s.strip())
    .replace({"": np.nan})
    .notna()
    .sum()
)
print(f"Mensagens com texto (coluna usada: '{CLEAN_COL}'):", int(n_clean))

# 4) Top remetentes (só se existir uma coluna compatível)
sender_candidates = ["remetente", "sender", "author", "from", "usuario", "user"]
SENDER_COL = next((cols_lower[c] for c in sender_candidates if c in cols_lower), None)

hr("Top 10 remetentes")
if SENDER_COL is not None:
    print(dfu[SENDER_COL].value_counts().head(10))
else:
    print("Coluna de remetente não encontrada; candidatos testados:", sender_candidates)



Estatísticas básicas
Mensagens totais: 1633
Mensagens com texto (coluna usada: 'texto'): 1633

Top 10 remetentes
Coluna de remetente não encontrada; candidatos testados: ['remetente', 'sender', 'author', 'from', 'usuario', 'user']


In [26]:
# CÉLULA 5 — Distribuição de intenções nos datasets
# Aqui inspecionamos os CSVs de intenções já construídos
# (curadobia_intents.csv e variações).

# Define os caminhos para os arquivos CSV de intenções.
p_train = INTENTS / "curadobia_intents.csv"
p_test = INTENTS / "curadobia_intents_test.csv"
p_reb = INTENTS / "curadobia_intents_rebalanced.csv"

def show_intents_dist(path, label):
    """
    Função auxiliar que lê um arquivo CSV de intenções,
    verifica se ele existe e exibe a distribuição de rótulos.
    """
    if path.exists():
        df = pd.read_csv(path)
        print(f"\n[{label}] {path} -> {df.shape}")
        # Exibe a contagem de cada rótulo na coluna 'label'.
        print(df["label"].value_counts())
    else:
        # Exibe uma mensagem de erro se o arquivo não for encontrado.
        print(f"[!] {path} não encontrado")

# Executa a função para cada um dos arquivos de intenções.
show_intents_dist(p_train, "TRAIN")
show_intents_dist(p_test, "TEST")
show_intents_dist(p_reb, "REBALANCED")


[TRAIN] c:\Users\win\Documents\GitHub\2025-2A-T07-CC11-G04\data\intents\curadobia_intents.csv -> (4107, 2)
label
nao_entendi                 2926
saudacao                     473
como_comprar                 242
agradecimento                173
tamanho_modelagem            166
frete_prazo                   42
formas_pagamento              36
erros_plataforma              33
troca_devolucao_politica      10
pedir_sugestao_produto         6
Name: count, dtype: int64

[TEST] c:\Users\win\Documents\GitHub\2025-2A-T07-CC11-G04\data\intents\curadobia_intents_test.csv -> (1027, 2)
label
nao_entendi                 732
saudacao                    119
como_comprar                 60
agradecimento                43
tamanho_modelagem            41
frete_prazo                  11
formas_pagamento              9
erros_plataforma              8
troca_devolucao_politica      2
pedir_sugestao_produto        2
Name: count, dtype: int64

[REBALANCED] c:\Users\win\Documents\GitHub\2025-2A-T07-CC11-G04\d

In [27]:
# CÉLULA 6 — Relatório de intenções
# Métricas obtidas com infer_intents.py e salvas em intents_report.json

# Define o caminho para o arquivo do relatório de intenções.
rep_path = RUNS / "intents_report.json"
# Verifica se o arquivo existe para evitar erros de leitura.
if rep_path.exists():
    # Carrega o conteúdo do arquivo JSON em um dicionário.
    rep = json.loads(rep_path.read_text(encoding="utf-8"))
    hr("Relatório de intenções")
    # Exibe o F1-macro, uma métrica importante para datasets desequilibrados.
    print("F1_macro:", rep.get("f1_macro"))
    # Exibe as métricas médias do relatório de classificação.
    print("\nMacro avg:", rep["report"].get("macro avg"))
    print("Weighted avg:", rep["report"].get("weighted avg"))
else:
    # Exibe uma mensagem de aviso se o arquivo não for encontrado.
    print("[!] Relatório não encontrado:", rep_path)


Relatório de intenções
F1_macro: 0.2459885151090301

Macro avg: {'precision': 0.4758549222797928, 'recall': 0.20541873016932238, 'f1-score': 0.2459885151090301, 'support': 1027.0}
Weighted avg: {'precision': 0.7470866904460398, 'recall': 0.7731256085686465, 'f1-score': 0.7049846333122957, 'support': 1027.0}


In [28]:
# CÉLULA 7 — Perfil da cliente
# Perfil traz medidas, estilos preferidos e restrições.

# Define o caminho para o arquivo JSON que contém o perfil da cliente.
perfil_path = PROF / "cliente_exemplo.json"
# Carrega o conteúdo do arquivo JSON para a variável 'perfil'.
perfil = json.loads(perfil_path.read_text(encoding="utf-8"))
# Imprime um separador e o título do perfil.
hr("Perfil da cliente")
# Exibe o perfil em um formato JSON amigável e legível.
pp(perfil)


Perfil da cliente
{
  "user_id": "demo",
  "tamanho_superior": "M",
  "tamanho_inferior": "38",
  "tamanhos_equivalentes": [
    "M",
    "38",
    "40"
  ],
  "estilos_preferidos": [
    "alfaiataria",
    "minimalista"
  ],
  "cores_evitar": [
    "amarelo"
  ],
  "ocasioes_frequentes": [
    "jantar",
    "trabalho"
  ],
  "tecidos_evitar": [
    "poliéster"
  ]
}


In [29]:
# CÉLULA 8 — Catálogo bruto
# Vamos visualizar o catálogo exportado (3 itens por enquanto).

# Define o caminho para o arquivo CSV do catálogo.
raw_catalog = CAT / "catalog_curadobia.csv"
# Carrega o catálogo em um DataFrame.
df_raw = pd.read_csv(raw_catalog)
# Imprime um separador e o título do catálogo.
hr("Catálogo bruto")
# Exibe o DataFrame completo do catálogo.
df_raw


Catálogo bruto


,id,brand,name,category,color,material,price,sizes,stock_json,description
0,101,Curadobia,Vestido Midi Luna,vestido,preto,viscose,299.9,P;M;G,"{""P"":1,""M"":2,""G"":0}",Vestido midi preto com caimento leve
1,102,Curadobia,Blazer Alfaiataria Ava,blazer,bege,linho,399.9,PP;P;M,"{""PP"":1,""P"":1,""M"":1}",Blazer de linho modelagem reta
2,103,Curadobia,Calça Reta Clara,calça,off-white,algodão,249.9,36;38;40,"{""36"":1,""38"":1,""40"":0}",Calça reta tecido encorpado


Após a etapa de preparação e separação dos dados, entramos no coração do nosso pipeline: a validação cruzada (CV). Enquanto a acurácia de um único teste nos dá uma pista sobre o desempenho do modelo, a validação cruzada oferece uma visão muito mais robusta e confiável.

Nesta seção, não nos contentamos com um único resultado. Nós dividimos os dados de treinamento em múltiplos subconjuntos (os "folds") para treinar e testar o modelo várias vezes. Ao fazer isso, conseguimos uma média de desempenho (F1-macro e acurácia) que é menos suscetível a pequenas variações nos dados, mostrando como o modelo se comporta de forma consistente. Esse processo nos ajuda a escolher o melhor modelo (Regressão Logística ou Random Forest) e a ter uma base sólida para comparar com futuras melhorias. A validação cruzada é a nossa garantia de que as nossas conclusões sobre o desempenho do modelo são estatisticamente válidas e não apenas um golpe de sorte.

In [30]:
# CÉLULA 9 — Normalização do catálogo
# Garante schema: id, brand, name, category, color, material, price, sizes, stock_json, description

# Define o caminho para o arquivo CSV do catálogo normalizado.
norm_catalog = CAT / "catalog_normalized.csv"
# Cria uma cópia do DataFrame bruto para evitar alterações no original.
df_norm = df_raw.copy()
# Normaliza os nomes das colunas, removendo espaços e convertendo para minúsculas.
df_norm.columns = [c.strip().lower() for c in df_norm.columns]
# Salva o DataFrame normalizado em um novo arquivo CSV, sem o índice.
df_norm.to_csv(norm_catalog, index=False)
# Imprime um separador e o título para a seção.
hr("Catálogo normalizado")
# Exibe o DataFrame normalizado.
df_norm


Catálogo normalizado


,id,brand,name,category,color,material,price,sizes,stock_json,description
0,101,Curadobia,Vestido Midi Luna,vestido,preto,viscose,299.9,P;M;G,"{""P"":1,""M"":2,""G"":0}",Vestido midi preto com caimento leve
1,102,Curadobia,Blazer Alfaiataria Ava,blazer,bege,linho,399.9,PP;P;M,"{""PP"":1,""P"":1,""M"":1}",Blazer de linho modelagem reta
2,103,Curadobia,Calça Reta Clara,calça,off-white,algodão,249.9,36;38;40,"{""36"":1,""38"":1,""40"":0}",Calça reta tecido encorpado


In [31]:
# CÉLULA 10 — Estatísticas do catálogo

# Imprime um separador e o título para a seção.
hr("Estatísticas do catálogo")
# Exibe o número total de itens no catálogo normalizado.
print("Itens:", len(df_norm))
# Exibe a contagem de cada categoria de produto em formato de dicionário.
print("Categorias:", df_norm["category"].value_counts().to_dict())
# Exibe a contagem de cada cor de produto em formato de dicionário.
print("Cores:", df_norm["color"].value_counts().to_dict())


Estatísticas do catálogo
Itens: 3
Categorias: {'vestido': 1, 'blazer': 1, 'calça': 1}
Cores: {'preto': 1, 'bege': 1, 'off-white': 1}


In [32]:
# CÉLULA 11 — Construção do índice semântico

from sentence_transformers import SentenceTransformer
import numpy as np

# Define e carrega o modelo de embeddings.
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedder = SentenceTransformer(model_name)

# Combina o nome e a descrição de cada produto em uma lista de textos para o embedder.
texts = (df_norm["name"].fillna("") + " " + df_norm["description"].fillna("")).tolist()
# Converte os textos em vetores numéricos (embeddings).
vectors = embedder.encode(texts, convert_to_numpy=True)

# Define o diretório para salvar os artefatos do índice e o cria, se necessário.
index_dir = MODELS / "catalog_index"
index_dir.mkdir(parents=True, exist_ok=True)
# Salva o catálogo, os vetores e os metadados do modelo no diretório do índice.
df_norm.to_csv(index_dir / "items.csv", index=False)
np.save(index_dir / "vectors.npy", vectors)
(index_dir / "meta.json").write_text(json.dumps({"model": model_name, "n_items": len(df_norm)}, indent=2), encoding="utf-8")

# Exibe informações sobre o índice criado para verificação.
hr("Índice criado")
print("Itens:", len(df_norm))
print("Index dir:", index_dir)


Índice criado
Itens: 3
Index dir: c:\Users\win\Documents\GitHub\2025-2A-T07-CC11-G04\code\notebooks\outputs\models\catalog_index


In [33]:
# CÉLULA 12 — Função de resposta contextual (versão melhorada)

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np, pandas as pd, json, re
from typing import List, Tuple

# Helpers ---------------------------------------------------------------

def _safe_lower(s):
    return str(s).lower().strip() if pd.notna(s) else ""

def _parse_sizes(cell) -> List[str]:
    """
    Aceita 'P;M;G' ou lista; devolve lista de strings normalizadas.
    """
    if isinstance(cell, (list, tuple, set)):
        return [str(x).strip() for x in cell if str(x).strip()]
    s = str(cell or "").strip()
    if not s:
        return []
    if ";" in s:
        return [x.strip() for x in s.split(";") if x.strip()]
    # fallback: separa por vírgula
    if "," in s:
        return [x.strip() for x in s.split(",") if x.strip()]
    return [s] if s else []

def _parse_stock_json(cell) -> dict:
    """
    Aceita dict já pronto ou string JSON do tipo {"P":1,"M":2,"G":0}.
    Devolve dict {tamanho: quantidade}.
    """
    if isinstance(cell, dict):
        return cell
    s = str(cell or "").strip()
    if not s:
        return {}
    try:
        return json.loads(s)
    except Exception:
        return {}

def _size_available_for_profile(row, tamanhos_equivalentes: List[str]) -> Tuple[bool, str]:
    """
    Verifica estoque real primeiro (stock_json), depois cai para 'sizes' como oferta.
    Retorna (tem_estoque, tamanho_compatível_encontrado_ou_"").
    """
    eq = [str(x).strip() for x in (tamanhos_equivalentes or []) if str(x).strip()]
    if not eq:
        return False, ""
    stock = _parse_stock_json(row.get("stock_json"))
    if stock:
        for t in eq:
            if str(stock.get(t, 0)).isdigit():
                if int(stock.get(t, 0)) > 0:
                    return True, t
            else:
                # caso venha "1"/"0" como string
                try:
                    if int(stock.get(t, "0")) > 0:
                        return True, t
                except Exception:
                    pass
    # fallback: apenas presença do tamanho no catálogo (sem garantia de estoque)
    sizes = set(_parse_sizes(row.get("sizes")))
    for t in eq:
        if t in sizes:
            return True, t
    return False, ""

def _style_boost(texto_item: str, estilos_preferidos: List[str]) -> float:
    """
    Bônus por palavras de estilo preferidas no título/descrição.
    """
    if not estilos_preferidos:
        return 0.0
    txt = _safe_lower(texto_item)
    score = 0.0
    for est in estilos_preferidos:
        est = _safe_lower(est)
        if est and re.search(rf"\b{re.escape(est)}\b", txt):
            score += 0.2  # cada estilo citado dá um pequeno boost
    return min(score, 0.6)  # cap

def _apply_constraints(row, perfil: dict) -> float:
    """
    Penalidades por cores/tecidos a evitar (se presentes).
    """
    desc = _safe_lower(row.get("description", ""))
    name = _safe_lower(row.get("name", ""))
    color = _safe_lower(row.get("color", ""))
    material = _safe_lower(row.get("material", ""))

    penalty = 0.0

    # cores a evitar
    for c in (perfil.get("cores_evitar") or []):
        c = _safe_lower(c)
        if c and (c in color or re.search(rf"\b{re.escape(c)}\b", desc)):
            penalty += 0.25

    # tecidos a evitar
    for t in (perfil.get("tecidos_evitar") or []):
        t = _safe_lower(t)
        if t and (t in material or re.search(rf"\b{re.escape(t)}\b", desc) or t in name):
            penalty += 0.25

    # ocasiões frequentes (leve bônus se mencionar)
    bonus = 0.0
    for oc in (perfil.get("ocasioes_frequentes") or []):
        oc = _safe_lower(oc)
        if oc and re.search(rf"\b{re.escape(oc)}\b", desc + " " + name):
            bonus += 0.1

    return bonus - penalty  # pode ser negativo

def _normalize_score(x: float) -> float:
    # só para manter score em uma faixa razoável
    return float(np.clip(x, -1.0, 2.0))

# Diversidade (MMR simplificado) ---------------------------------------

def _mmr_select(indices_sorted: List[int],
                item_vectors: np.ndarray,
                q_vec: np.ndarray,
                lambda_diversity: float,
                topk: int) -> List[int]:
    """
    Seleção greedy estilo MMR.
    indices_sorted: índices ordenados por score atual (desc).
    item_vectors: matriz (n_items, dim).
    q_vec: vetor da query (dim,).
    lambda_diversity: 0.0 (sem diversidade) a 1.0 (só diversidade).
    topk: quantos selecionar.
    """
    if topk <= 0 or len(indices_sorted) == 0:
        return []
    selected = []
    candidates = indices_sorted.copy()

    # Pré-computar similaridades item-item para diversidade
    # (usando cosseno; pode ser pesado em catálogos grandes — ok para demo atual)
    if len(candidates) > 1:
        item_norm = item_vectors / (np.linalg.norm(item_vectors, axis=1, keepdims=True) + 1e-9)
        sim_item_item = item_norm @ item_norm.T
    else:
        sim_item_item = None

    while candidates and len(selected) < topk:
        if not selected:
            pick = candidates[0]
            selected.append(pick)
            candidates.pop(0)
            continue

        # Para cada candidato, calcula: lambda * sim(q, i) - (1-lambda) * max_j sim(i, j)
        best_idx = None
        best_score = -1e9
        for idx in candidates:
            # similaridade com a query ~ já está refletida nos scores iniciais,
            # mas aqui usamos o dot com q_vec normalizado como aproximação.
            s_q = float(cosine_similarity(item_vectors[idx:idx+1], q_vec[None, :])[0, 0])

            # penalidade por similaridade com já selecionados
            if sim_item_item is not None:
                s_div = max(sim_item_item[idx, s] for s in selected)
            else:
                s_div = 0.0

            mmr = lambda_diversity * s_q - (1.0 - lambda_diversity) * s_div
            if mmr > best_score:
                best_score = mmr
                best_idx = idx
        selected.append(best_idx)
        candidates.remove(best_idx)

    return selected

# Função principal ------------------------------------------------------

def responder(query: str,
              perfil: dict,
              topk: int = 5,
              debug: bool = False,
              w_sim: float = 1.0,
              w_tam: float = 1.0,
              w_estilo: float = 1.0,
              w_rules: float = 1.0,
              lambda_diversity: float = 0.2) -> List[Tuple[pd.Series, float, float, float, float]]:
    """
    Recomenda itens combinando:
      - Similaridade semântica (embeddings) [w_sim]
      - Tamanho/estoque compatível [w_tam]
      - Estilos preferidos [w_estilo]
      - Regras de perfil (cores/tecidos a evitar, ocasiões) [w_rules]
      - Diversidade via MMR (lambda_diversity)

    Retorna lista de tuplas: (row, score_total, score_sim, score_tam, score_estilo)
    """
    # Carrega índice
    df_items = pd.read_csv(index_dir / "items.csv")
    vecs = np.load(index_dir / "vectors.npy")
    if len(df_items) == 0:
        return []

    # Embed da query
    q_vec = embedder.encode([query], convert_to_numpy=True)
    sims = cosine_similarity(q_vec, vecs)[0]  # shape (n_items,)

    # Preparos do perfil
    tamanhos_eq = perfil.get("tamanhos_equivalentes") or []
    estilos_pref = perfil.get("estilos_preferidos") or []

    # Calcula scores
    scored = []
    for i, row in df_items.iterrows():
        # sim semântica
        s_sim = float(sims[i])

        # tamanho/estoque
        tem, t_compat = _size_available_for_profile(row, tamanhos_eq)
        s_tam = 1.0 if tem else 0.0

        # estilo preferido (usa nome+descrição)
        txt_item = f"{row.get('name','')} {row.get('description','')}"
        s_est = _style_boost(txt_item, estilos_pref)

        # regras/penalidades (cores/tecidos a evitar, bônus ocasião)
        s_rules = _apply_constraints(row, perfil)

        # combinação
        score = (
            w_sim * s_sim +
            w_tam * s_tam +
            w_estilo * s_est +
            w_rules * s_rules
        )
        scored.append((i, _normalize_score(score), s_sim, s_tam, s_est))

    # Ordena por score total
    scored.sort(key=lambda x: x[1], reverse=True)

    # Diversidade (MMR) opcional: reordena os 'indices' por MMR e depois mapeia de volta
    indices_sorted = [i for (i, _, _, _, _) in scored]
    if lambda_diversity is not None and 0.0 <= lambda_diversity <= 1.0 and len(indices_sorted) > 1:
        sel = _mmr_select(indices_sorted, vecs, q_vec.squeeze(0), lambda_diversity, topk)
        # Monta lista final seguindo a ordem selecionada
        picked = []
        sc_map = {i: (score, s_sim, s_tam, s_est) for (i, score, s_sim, s_tam, s_est) in scored}
        for idx in sel:
            sc, s_sim, s_tam, s_est = sc_map[idx]
            picked.append((idx, sc, s_sim, s_tam, s_est))
    else:
        picked = scored[:topk]

    # Constrói saída (mesmo formato usado nas células seguintes)
    results = []
    for (i, sc, s_sim, s_tam, s_est) in picked[:topk]:
        row = df_items.iloc[i]
        results.append((row, sc, s_sim, s_tam, s_est))

    if debug:
        for (row, sc, s_sim, s_tam, s_est) in results:
            print(f"{row['name']} | score_total={sc:.3f} (sim={s_sim:.3f}, tam={s_tam:.1f}, est={s_est:.2f}, rules={_normalize_score(sc - (w_sim*s_sim + w_tam*s_tam + w_estilo*s_est)):+.2f})")

    return results


In [34]:
# CÉLULA 13 — Testes de queries

# Define uma lista de queries para testar o sistema de recomendação.
queries = [
    "Quero um vestido midi preto para um jantar.",
    "Preciso de um blazer de linho para o trabalho.",
    "Busco uma calça reta off-white para escritório."
]

# Itera sobre cada query e executa o sistema de recomendação.
for q in queries:
    # Imprime a query atual usando um separador.
    hr(f"Query: {q}")
    # Chama a função 'responder' para obter as 3 principais recomendações.
    # O 'debug=True' exibe os scores de similaridade, tamanho e estilo para cada item.
    res = responder(q, perfil, topk=3, debug=True)
    print("Sugestões finais:")
    # Itera sobre os resultados e exibe as informações principais de cada item.
    for row, score_total, sim, st, se in res:
        print(f" - {row['name']} ({row['category']}, {row['color']}, {row['material']}) — R$ {row['price']}")


Query: Quero um vestido midi preto para um jantar.
Vestido Midi Luna | score_total=1.431 (sim=0.431, tam=1.0, est=0.00, rules=+0.00)
Blazer Alfaiataria Ava | score_total=1.388 (sim=0.188, tam=1.0, est=0.20, rules=+0.00)
Calça Reta Clara | score_total=1.229 (sim=0.229, tam=1.0, est=0.00, rules=+0.00)
Sugestões finais:
 - Vestido Midi Luna (vestido, preto, viscose) — R$ 299.9
 - Blazer Alfaiataria Ava (blazer, bege, linho) — R$ 399.9
 - Calça Reta Clara (calça, off-white, algodão) — R$ 249.9

Query: Preciso de um blazer de linho para o trabalho.
Blazer Alfaiataria Ava | score_total=1.575 (sim=0.375, tam=1.0, est=0.20, rules=+0.00)
Vestido Midi Luna | score_total=1.140 (sim=0.140, tam=1.0, est=0.00, rules=+0.00)
Calça Reta Clara | score_total=1.346 (sim=0.346, tam=1.0, est=0.00, rules=+0.00)
Sugestões finais:
 - Blazer Alfaiataria Ava (blazer, bege, linho) — R$ 399.9
 - Vestido Midi Luna (vestido, preto, viscose) — R$ 299.9
 - Calça Reta Clara (calça, off-white, algodão) — R$ 249.9

Quer

In [35]:
# CÉLULA 14 — Salvar ranking em CSV

# Define o caminho de saída para o arquivo CSV.
out_csv = RUNS / "context_ranking_samples.csv"
# Inicializa uma lista para armazenar os dados de todas as recomendações.
all_rows = []

# Itera sobre cada query de teste para obter as recomendações.
for q in queries:
    # Chama a função 'responder' para obter os 5 melhores resultados.
    res = responder(q, perfil, topk=5)
    # Itera sobre os resultados, extraindo os dados de cada recomendação.
    for rank, (row, score_total, sim, st, se) in enumerate(res, 1):
        # Adiciona um dicionário com os detalhes da recomendação à lista.
        all_rows.append({
            "query": q, "rank": rank,
            "name": row["name"], "category": row["category"], "color": row["color"],
            "material": row["material"], "price": row["price"], "sizes": row["sizes"],
            "score_total": score_total, "sim": sim, "score_tamanho": st, "score_estilo": se
        })

# Cria um DataFrame a partir da lista de dicionários e o salva em um arquivo CSV.
pd.DataFrame(all_rows).to_csv(out_csv, index=False)
# Exibe uma mensagem de confirmação e o caminho do arquivo salvo.
hr("Ranking salvo")
print(out_csv)


Ranking salvo
c:\Users\win\Documents\GitHub\2025-2A-T07-CC11-G04\code\notebooks\outputs\runs\context_ranking_samples.csv


A nossa jornada não termina com o modelo treinado. Um modelo "bom" não é apenas aquele que acerta a maioria das vezes, mas sim aquele que sabe quando não sabe. Essa é a essência do que chamamos de "gating por confiança".

Depois de treinar o modelo, o avaliamos para além da sua precisão. Analisamos a sua confiança em cada previsão. Se a confiança for baixa, ou se a diferença entre as duas melhores previsões for muito pequena (o que chamamos de top-2 gap), o modelo não "chuta" uma resposta. Em vez disso, ele atribui um rótulo de fallback, como "não entendi", sinalizando que a incerteza é alta. Este é um passo crucial para tornar o nosso assistente mais seguro e confiável em interações reais com clientes. O notebook documenta de forma explícita como esses parâmetros (threshold e gap) são otimizados, e como eles impactam o desempenho do modelo, garantindo um processo transparente e reprodutível.

In [36]:
# CÉLULA 15 — Visualização do ranking salvo

# Lê o arquivo CSV com o ranking de recomendações.
df_rank = pd.read_csv(out_csv)
# Imprime um separador e o título da seção.
hr("Ranking contextual")
# Exibe as 15 primeiras linhas do DataFrame para inspeção.
df_rank.head(15)


Ranking contextual


,query,rank,name,category,color,material,price,sizes,score_total,sim,score_tamanho,score_estilo
0,Quero um vestido midi preto para um jantar.,1,Vestido Midi Luna,vestido,preto,viscose,299.9,P;M;G,1.430514,0.430514,1.0,0.0
1,Quero um vestido midi preto para um jantar.,2,Blazer Alfaiataria Ava,blazer,bege,linho,399.9,PP;P;M,1.387948,0.187948,1.0,0.2
2,Quero um vestido midi preto para um jantar.,3,Calça Reta Clara,calça,off-white,algodão,249.9,36;38;40,1.229073,0.229073,1.0,0.0
3,Preciso de um blazer de linho para o trabalho.,1,Blazer Alfaiataria Ava,blazer,bege,linho,399.9,PP;P;M,1.575148,0.375148,1.0,0.2
4,Preciso de um blazer de linho para o trabalho.,2,Vestido Midi Luna,vestido,preto,viscose,299.9,P;M;G,1.140303,0.140303,1.0,0.0
5,Preciso de um blazer de linho para o trabalho.,3,Calça Reta Clara,calça,off-white,algodão,249.9,36;38;40,1.345789,0.345789,1.0,0.0
6,Busco uma calça reta off-white para escritório.,1,Vestido Midi Luna,vestido,preto,viscose,299.9,P;M;G,1.314945,0.314945,1.0,0.0
7,Busco uma calça reta off-white para escritório.,2,Blazer Alfaiataria Ava,blazer,bege,linho,399.9,PP;P;M,1.298043,0.098043,1.0,0.2
8,Busco uma calça reta off-white para escritório.,3,Calça Reta Clara,calça,off-white,algodão,249.9,36;38;40,1.305651,0.305651,1.0,0.0


In [37]:
# CÉLULA 16 — Estatísticas do ranking

# Imprime um separador e o título da seção.
hr("Estatísticas do ranking")
# Exibe o número de queries únicas que foram processadas.
print("Queries distintas:", df_rank["query"].nunique())
# Exibe o número de itens distintos que foram retornados nas recomendações.
print("Itens retornados:", df_rank["name"].nunique())
# Exibe a distribuição de ranks, mostrando a frequência de cada posição.
print("Distribuição de ranks:")
print(df_rank["rank"].value_counts().sort_index())


Estatísticas do ranking
Queries distintas: 3
Itens retornados: 3
Distribuição de ranks:
rank
1    3
2    3
3    3
Name: count, dtype: int64


In [38]:
# CÉLULA 16B — Estatísticas do ranking

# Imprime um separador e o título da seção.
hr("Estatísticas do ranking")
# Exibe o número de queries únicas que foram processadas.
print("Queries distintas:", df_rank["query"].nunique())
# Exibe o número de itens distintos que foram retornados nas recomendações.
print("Itens retornados:", df_rank["name"].nunique())
# Exibe a distribuição de ranks, mostrando a frequência de cada posição.
print("Distribuição de ranks:")
print(df_rank["rank"].value_counts().sort_index())


Estatísticas do ranking
Queries distintas: 3
Itens retornados: 3
Distribuição de ranks:
rank
1    3
2    3
3    3
Name: count, dtype: int64


In [39]:
# CÉLULA 17 — Conexão com intenções
# Mostra como intenções específicas poderiam acionar este sistema.

# Define uma lista de intenções de exemplo que poderiam acionar o sistema de recomendação.
intents_exemplo = ["pedir_sugestao_produto", "tamanho_modelagem"]
print("Exemplo de intenções que acionariam recomendações contextuais:")
# Itera sobre a lista e exibe cada intenção.
for i in intents_exemplo:
    print(" -", i)

Exemplo de intenções que acionariam recomendações contextuais:
 - pedir_sugestao_produto
 - tamanho_modelagem


In [40]:
# CÉLULA 18 — Clusters de mensagens (prévia do Item 3)
# Aqui apenas inspecionamos arquivos já gerados em runs.

# Define o caminho para o arquivo CSV de clusters.
clust_csv = RUNS / "clusters_questions.csv"
# Verifica se o arquivo de clusters existe.
if clust_csv.exists():
    # Lê o arquivo CSV e o carrega em um DataFrame.
    dfc = pd.read_csv(clust_csv)
    # Imprime um separador e o título da seção.
    hr("Clusters")
    # Exibe o número de clusters distintos encontrados no dataset.
    print("Clusters distintos:", dfc["cluster"].nunique())
    # Mostra as 10 primeiras linhas do DataFrame de clusters para inspeção.
    dfc.head(10)


Clusters
Clusters distintos: 14


In [41]:
# CÉLULA 19 — Top termos por cluster (corrigida)

clusters_csv = RUNS / "clusters_questions.csv"

# Verifica se o arquivo CSV de clusters existe no diretório 'RUNS'.
if clusters_csv.exists():
    dfc = pd.read_csv(clusters_csv)
    if "texto" in dfc.columns:
        from sklearn.feature_extraction.text import TfidfVectorizer
        # Define uma lista básica de stopwords em português.
        stop_pt = {"a", "o", "as", "os", "de", "do", "da", "das", "dos", "e", "é", "ser", "em", "um", "uma", "para", "pra", "por", "com"}
        dfc["_clean"] = dfc["texto"].fillna("").astype(str)
        
        # Inicializa o vetorizador TF-IDF para converter o texto em vetores numéricos.
        # Ele limita o vocabulário a 2000 termos e usa as stopwords definidas.
        vect = TfidfVectorizer(max_features=2000, stop_words=list(stop_pt))
        X = vect.fit_transform(dfc["_clean"])
        
        print("Top termos globais:")
        # Calcula a soma dos pesos de cada termo na matriz TF-IDF para encontrar os mais importantes.
        sums = np.array(X.sum(axis=0)).ravel()
        # Mapeia cada termo ao seu peso total.
        terms = [(t, sums[i]) for t, i in vect.vocabulary_.items()]
        # Ordena os termos por peso de forma decrescente e seleciona os 15 primeiros.
        terms = sorted(terms, key=lambda x: x[1], reverse=True)[:15]
        print(terms)
    else:
        # Mensagem de erro se a coluna 'texto' não for encontrada.
        print("⚠ Coluna 'texto' não encontrada em clusters_questions.csv")
else:
    # Mensagem de erro se o arquivo CSV de clusters não for encontrado.
    print("⚠ clusters_questions.csv não encontrado em:", clusters_csv)

Top termos globais:
[('que', np.float64(77.9400800229677)), ('num', np.float64(67.47636856931989)), ('obrigada', np.float64(55.57845020909645)), ('ola', np.float64(41.877588493625)), ('tudo', np.float64(41.799707914029796)), ('bem', np.float64(41.4512235155064)), ('no', np.float64(40.923287739970924)), ('aqui', np.float64(38.05804428336405)), ('curadobia', np.float64(35.28336532722843)), ('bom', np.float64(34.88815068317756)), ('na', np.float64(34.4994481848977)), ('dia', np.float64(32.38839543725493)), ('qualquer', np.float64(29.557650210959903)), ('mas', np.float64(26.31017310441776)), ('estamos', np.float64(25.449320594230148))]



# ✅ 

 - Perfil da cliente (carregado do JSON)
 - Catálogo normalizado e indexado
 - Sistema de respostas contextuais (função + ranking)
 - Export de resultados para auditoria

 Além disso, adicionamos análises intermediárias
 (dataset, intenções, clusters) para fortalecer a entrega.


Este notebook nos permitiu ir além de um modelo de classificação simples, construindo um sistema mais robusto, auditável e pronto para produção. O feedback sobre a necessidade de um caderno mais explicativo e reprodutível foi a nossa bússola. Nós não apenas treinamos o modelo, mas também documentamos cada etapa de forma a ser compreensível para qualquer pessoa, com explicações sobre a lógica de decisões, gráficos e métricas de interpretabilidade.

Os artefatos gerados, como o modelo calibrado, os thresholds de confiança e os relatórios de avaliação, estão prontos para serem integrados em um ambiente de produção. Além disso, a publicação do modelo no Hugging Face Hub, e a nossa preocupação em usar formatos de arquivo mais seguros como o safetensors (mencionado no feedback) em futuras sprints, são passos em direção a um pipeline de MLOps cada vez mais maduro. Este trabalho estabelece uma base sólida para a próxima fase, onde focaremos na expansão da base de conhecimento e na construção de fluxos de conversação mais dinâmicos.